# ExploreKit — офлайн-оценка политик (OPE)

Демонстрация модуля участника 2. Вся логика — в `explorekit/ope/`, здесь только сценарий использования.

**Перед запуском:** если нет данных, выполните `bash scripts/download_obd.sh`

## Что показываем
1. Загрузка Open Bandit Dataset в формат проекта
2. Обучение reward model
3. Оценка трёх политик exploration: IPS / SNIPS / DR + 95% CI
4. Диагностика надёжности
5. Валидация на синтетике с известной истиной

In [1]:
import sys
from pathlib import Path

# запуск из папки notebooks/ — поднимаемся в корень репозитория
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml

config = yaml.safe_load((ROOT / 'configs' / 'ope.yaml').read_text())
SEED = config['seed']
MAX_WEIGHT = config['ope']['max_weight']
config

{'seed': 42,
 'ope': {'max_weight': 15.0},
 'bootstrap': {'n_bootstrap': 1000, 'confidence_level': 0.95},
 'reward_model': {'C': 1.0, 'max_iter': 1000, 'train_fraction': 0.5},
 'diagnostics': {'ess_ratio_caution': 0.2,
  'ess_ratio_unreliable': 0.05,
  'clipped_fraction_caution': 0.05,
  'support_violation_caution': 0.01},
 'synthetic_validation': {'n_replications': 50,
  'n_rounds': 5000,
  'n_bootstrap': 200,
  'epsilon': 0.1},
 'open_bandit': {'data_dir': 'data/obd',
  'campaign': 'all',
  'behavior_policy': 'random',
  'evaluation_epsilons': {'conservative': 0.05,
   'moderate': 0.1,
   'aggressive': 0.25}}}

## 1. Загрузка Open Bandit Dataset

In [3]:
from explorekit.datasets.open_bandit import load_open_bandit_dataset

dataset = load_open_bandit_dataset(
    data_dir=ROOT / config['open_bandit']['data_dir'],
    campaign=config['open_bandit']['campaign'],
    behavior_policy=config['open_bandit']['behavior_policy'],
)
print(f'раундов: {len(dataset)}, действий: {dataset.n_actions}')
print(f"CTR логирующей политики: {dataset.logs['reward'].mean():.4%}")
dataset.logs.head()

раундов: 10000, действий: 80
CTR логирующей политики: 0.3800%


,timestamp,request_id,chosen_item,propensity,reward,position,policy_name
0,2019-11-24 00:00:34.762830+00:00,0,14,0.0125,0.0,3,obd_random
1,2019-11-24 00:00:53.965051+00:00,1,14,0.0125,0.0,3,obd_random
2,2019-11-24 00:00:56.727734+00:00,2,27,0.0125,0.0,3,obd_random
3,2019-11-24 00:02:17.189232+00:00,3,48,0.0125,0.0,2,obd_random
4,2019-11-24 00:03:02.129117+00:00,4,36,0.0125,0.0,2,obd_random


## 2. Reward model

Обучаем на одной половине логов. Вторая остаётся чистой для оценки — иначе DR подглядит в собственные обучающие данные и оценка будет оптимистично смещена.

In [4]:
from explorekit.ope import RewardModel, split_for_reward_model

n = len(dataset)
train_idx, eval_idx = split_for_reward_model(
    n, train_fraction=config['reward_model']['train_fraction'], seed=SEED
)

chosen = dataset.logs['chosen_item'].to_numpy(dtype=int)
chosen_features = dataset.action_features[np.arange(n), chosen]
reward = dataset.logs['reward'].to_numpy(dtype=float)

reward_model = RewardModel(seed=SEED).fit(
    context=dataset.context[train_idx],
    action_features_chosen=chosen_features[train_idx],
    reward=reward[train_idx],
    n_actions=dataset.n_actions,
)

eval_logs = dataset.logs.iloc[eval_idx].reset_index(drop=True)
q_hat = reward_model.predict_all_actions(
    dataset.context[eval_idx], dataset.action_features[eval_idx]
)
print(f'обучено на {len(train_idx)}, оцениваем на {len(eval_idx)}')
print(f'q_hat: {q_hat.shape}')

обучено на 5000, оцениваем на 5000
q_hat: (5000, 80)


## 3. Оценка политик exploration

Используется `EpsilonGreedyPolicy` из `explorekit/policies/epsilon_greedy.py` — реальная политика участника 1.

Контракт совместим с `ActionDistributionFn` из `explorekit/ope/base.py`:

In [5]:
from explorekit.ope import (
    DREstimator,
    IPSEstimator,
    SNIPSEstimator,
    build_ope_input,
    compute_reliability,
)

from explorekit.policies.epsilon_greedy import EpsilonGreedyPolicy

base_scores = q_hat  # base ranker = скор reward model
n_rounds = base_scores.shape[0]
n_actions = base_scores.shape[1]

# Контекст: в OBD это user features. Для epsilon-greedy он не используется,
# но передаём, чтобы соблюсти контракт BasePolicy.
context_for_ope = np.zeros((n_rounds, 1))

estimators = [
    IPSEstimator(max_weight=MAX_WEIGHT),
    SNIPSEstimator(max_weight=MAX_WEIGHT),
    DREstimator(max_weight=MAX_WEIGHT),
]

rows = []
for name, eps in config['open_bandit']['evaluation_epsilons'].items():
    # Настоящая политика из модуля участника 1
    policy = EpsilonGreedyPolicy(
        n_actions=n_actions,
        epsilon=eps,
        seed=SEED,
    )
    dist = policy.action_distribution(context_for_ope, base_scores)

    ope_input = build_ope_input(
        logs=eval_logs,
        evaluation_action_dist=dist,
        policy_name=f'{name}(eps={eps})',
        q_hat_all_actions=q_hat,
    )
    diag = compute_reliability(ope_input, max_weight=MAX_WEIGHT)

    for est in estimators:
        res = est.estimate_with_ci(ope_input, n_bootstrap=500, seed=SEED)
        rows.append({
            'policy': name, 'epsilon': eps, 'estimator': res.estimator_name,
            'CTR': res.estimate, 'ci_low': res.ci_low, 'ci_high': res.ci_high,
            'ESS/N': diag.ess_ratio, 'status': diag.status.value,
        })

results = pd.DataFrame(rows)
results

,policy,epsilon,estimator,CTR,ci_low,ci_high,ESS/N,status
0,conservative,0.05,IPS(clip=15),0.000150,0.000080,0.000220,0.011543,Unreliable
1,conservative,0.05,SNIPS(clip=15),0.000741,0.000370,0.001156,0.011543,Unreliable
2,conservative,0.05,DR(clip=15),0.003714,0.003462,0.003934,0.011543,Unreliable
3,moderate,0.10,IPS(clip=15),0.000300,0.000160,0.000440,0.013128,Unreliable
4,moderate,0.10,SNIPS(clip=15),0.001191,0.000609,0.001825,0.013128,Unreliable
5,moderate,0.10,DR(clip=15),0.003639,0.003348,0.003883,0.013128,Unreliable
6,aggressive,0.25,IPS(clip=15),0.000750,0.000400,0.001100,0.020034,Unreliable
7,aggressive,0.25,SNIPS(clip=15),0.001873,0.000977,0.002830,0.020034,Unreliable
8,aggressive,0.25,DR(clip=15),0.003416,0.002972,0.003844,0.020034,Unreliable


### Диагностика надёжности

На сэмпле OBD всего 38 кликов на 10 000 показов, поэтому ESS обваливается и модуль честно помечает оценки как ненадёжные. Это ожидаемый результат, а не сбой: для отчёта нужен полный датасет.

In [6]:
from explorekit.policies.epsilon_greedy import EpsilonGreedyPolicy

eps = config['open_bandit']['evaluation_epsilons']['moderate']
n_rounds = base_scores.shape[0]
n_actions = base_scores.shape[1]

# Контекст для epsilon-greedy не используется, но соблюдаем контракт.
context_for_ope = np.zeros((n_rounds, 1))

policy = EpsilonGreedyPolicy(
    n_actions=n_actions,
    epsilon=eps,
    seed=SEED,
)
dist = policy.action_distribution(context_for_ope, base_scores)

ope_input = build_ope_input(
    logs=eval_logs,
    evaluation_action_dist=dist,
    policy_name=f'moderate(eps={eps})',
    q_hat_all_actions=q_hat,
)
diag = compute_reliability(ope_input, max_weight=MAX_WEIGHT)
print(f'Статус: {diag.status.value}')
print(f'Причина: {diag.explanation}')
pd.Series(diag.to_dict())

Статус: Unreliable
Причина: ESS/N = 1.3% — оценка фактически держится на 66 эффективных наблюдениях из 5000; политики почти не перекрываются.


status                                                           Unreliable
explanation               ESS/N = 1.3% — оценка фактически держится на 6...
ess                                                               65.639737
ess_ratio                                                          0.013128
max_weight                                                             72.1
mean_weight                                                          0.8344
weight_variance                                                   52.347926
weight_cv                                                          8.671125
weight_p99                                                             72.1
ips_standard_error                                                 0.000077
fraction_clipped                                                     0.0102
support_violations                                                        0
support_violation_rate                                                  0.0
n_rounds    

## 4. Валидация на синтетике с известной истиной

На реальных данных истинный CTR новой политики неизвестен, поэтому проверить оценщики там нельзя в принципе. На синтетике мы сами задали P(click | user, item) и знаем правильный ответ точно.

In [7]:
from functools import partial

from experiments._stub_environment import make_replication
from explorekit.ope.validation import run_single_comparison, run_validation

sv = config['synthetic_validation']
replication_fn = partial(
    make_replication, n_rounds=sv['n_rounds'], epsilon=sv['epsilon']
)

single = replication_fn(SEED)
print(f'истинный CTR политики: {single.true_value:.4%}')
run_single_comparison(single, estimators, n_bootstrap=500, seed=SEED)

истинный CTR политики: 5.2802%


,estimator,true_ctr,estimate,abs_error,ci_low,ci_high,covers_true
0,IPS(clip=15),0.052802,0.047959,0.004842,0.036690,0.061284,True
1,SNIPS(clip=15),0.052802,0.048021,0.004781,0.036567,0.059787,True
2,DR(clip=15),0.052802,0.049562,0.003240,0.038141,0.061076,True


### Bias / Std / RMSE / CI coverage

CI coverage — главная строка: если интервал заявлен как 95%, он и должен накрывать истину примерно в 95% прогонов.

In [8]:
report = run_validation(
    replication_fn, estimators,
    n_replications=sv['n_replications'],
    n_bootstrap=sv['n_bootstrap'],
    seed=SEED,
)
print(report.format_table())
report.to_frame()

Оценщик            Среднее       Bias       Std      RMSE   CI cover   CI width
-------------------------------------------------------------------------------
IPS(clip=15)        0.0510    -0.0004    0.0067    0.0064     96.0%     0.0250
SNIPS(clip=15)      0.0506    -0.0007    0.0064    0.0062     98.0%     0.0246
DR(clip=15)         0.0510    -0.0004    0.0064    0.0062     94.0%     0.0242


,estimator_name,n_replications,true_value,mean_estimate,bias,std,rmse,mean_abs_error,ci_coverage,mean_ci_width
0,IPS(clip=15),50,0.05138,0.050999,-0.000381,0.006660,0.006410,0.005386,0.96,0.025041
1,SNIPS(clip=15),50,0.05138,0.050649,-0.000731,0.006440,0.006189,0.005113,0.98,0.024593
2,DR(clip=15),50,0.05138,0.050968,-0.000412,0.006372,0.006156,0.005010,0.94,0.024186


## 5. Сверка с эталонной реализацией Open Bandit Pipeline

Собственные тесты проверяют формулы против моих же ожиданий. Сверка с независимой реализацией авторов датасета ловит ошибки, которые такие тесты пропустили бы по построению.

In [9]:
from experiments._obp_crosscheck import obp_available

if not obp_available:
    print('obp недоступен — сверка пропущена (pip install obp)')
else:
    from experiments._obp_crosscheck import (
        reference_dr,
        reference_ips,
        reference_snips,
    )
    action = eval_logs['chosen_item'].to_numpy(dtype=int)
    pscore = eval_logs['propensity'].to_numpy(dtype=float)
    rew = eval_logs['reward'].to_numpy(dtype=float)
    comparison = pd.DataFrame([
        {'estimator': 'IPS',
         'ours': IPSEstimator().estimate(ope_input),
         'obp': reference_ips(rew, action, pscore, dist)},
        {'estimator': 'SNIPS',
         'ours': SNIPSEstimator().estimate(ope_input),
         'obp': reference_snips(rew, action, pscore, dist)},
        {'estimator': 'DR',
         'ours': DREstimator().estimate(ope_input),
         'obp': reference_dr(rew, action, pscore, dist, q_hat)},
    ])
    comparison['diff'] = (comparison['ours'] - comparison['obp']).abs()
    display(comparison)

,estimator,ours,obp,diff
0,IPS,0.000300,0.000300,0.000000e+00
1,SNIPS,0.000360,0.000360,0.000000e+00
2,DR,0.000783,0.000783,1.192622e-18


## Выводы

- Формулы IPS / SNIPS / DR совпадают с эталонной реализацией до машинной точности.
- На синтетике смещение всех трёх оценщиков около нуля, CI coverage близок к заявленным 95%, у DR наименьший RMSE.
- На сэмпле OBD оценки честно помечаются как ненадёжные: 38 кликов недостаточно, нужен полный датасет (26 млн показов).

Все параметры — в `configs/ope.yaml`, результат воспроизводим с seed.